In [ ]:
import numpy as np
import pandas as pd
from scipy.special import gammaln, betaln, hyp2f1
print("Todo bien")

In [ ]:
ruta = "../data/raw/CDNOW_sample.txt"

df = pd.read_csv(
    ruta,
    sep=r"\s+",
    header=None,
    names=["id_master", "id", "fecha", "cds", "monto"]
)

df["fecha"] = pd.to_datetime(df["fecha"], format="%Y%m%d")

print(df.shape)
print(df["id"].nunique())
df.head()

In [ ]:
# 0.1 Unir compras del mismo día en una sola transacción
duplicadas = df.duplicated(subset=["id", "fecha"]).sum()
print(duplicadas)   # 223

df = df.groupby(["id", "fecha"]).agg({"cds": "sum", "monto": "sum"}).reset_index()
print(df.shape)     # (6696, 4)

# 0.2 Tiempo en semanas, con día 0 = 1 de enero de 1997
df["dias"] = (df["fecha"] - pd.Timestamp("1997-01-01")).dt.days
df["semana"] = df["dias"] / 7
df.head()

In [ ]:
fecha_fin_cal = pd.Timestamp("1997-09-30")     # último día de calibración, inclusive

cal = df[df["fecha"] <= fecha_fin_cal]
hold = df[df["fecha"] > fecha_fin_cal]

print(cal.shape, hold.shape)
print(cal["id"].nunique())   # ¿cuántos clientes deben salir?

In [ ]:
# 0.3 Calibración y holdout
fecha_fin_cal = pd.Timestamp("1997-09-30")      # último día de calibración, inclusive

cal  = df[df["fecha"] <= fecha_fin_cal]
hold = df[df["fecha"] >  fecha_fin_cal]

# Resumen por cliente con los datos de calibración
resumen = cal.groupby("id")["fecha"].agg(primera="min", ultima="max", n="count").reset_index()

resumen["x"]   = resumen["n"] - 1                                            # compras repetidas
resumen["t_x"] = (resumen["ultima"] - resumen["primera"]).dt.days / 7        # última compra, desde la primera
resumen["T"]   = (fecha_fin_cal - resumen["primera"]).dt.days / 7            # primera compra → fin de calibración

# x*: compras en el holdout (0 si no compró)
x_star = hold.groupby("id").size().rename("x_star")
resumen = resumen.merge(x_star, on="id", how="left")
resumen["x_star"] = resumen["x_star"].fillna(0).astype(int)

# Verificaciones
assert len(resumen) == 2357
assert resumen["x_star"].sum() == 1882
print(resumen[["x", "t_x", "T", "x_star"]].describe())

# 0.4 Fracción que no volvió a comprar en la calibración
frac_sin_recompra = (resumen["x"] == 0).mean()
print(f"Fracción con x = 0: {frac_sin_recompra:.1%}")
print(f"T mínimo: {resumen['T'].min():.3f}, T máximo: {resumen['T'].max():.3f}")

resumen.head()

**1. Fracción de clientes sin recompras**

El 59.86% de los clientes no volvió a comprar durante la calibración ($x = 0$).

**2. Por qué $T$ es distinto para cada cliente**

$T$ es el tiempo que cada cliente estuvo bajo observación: va desde su primera compra hasta el fin de la calibración (30 de septiembre de 1997). Como los clientes de la cohorte hicieron su primera compra en fechas distintas, no todos tuvieron el mismo tiempo para volver a comprar:

- **Máximo (38.86 semanas):** clientes que compraron el 1 de enero de 1997.
- **Mínimo (27.00 semanas):** clientes que compraron por primera vez el 25 de marzo de 1997.

Esto importa porque un cliente con más tiempo observado tuvo más oportunidades de comprar, así que comparar $x$ sin considerar $T$ sería injusto. Por eso el modelo usa $(x, t_x, T)$ y no solo la frecuencia.

In [ ]:
import pandas as pd

FECHA_INICIO = pd.Timestamp("1997-01-01")
FECHA_FIN_CAL = pd.Timestamp("1997-09-30")


def cargar_transacciones(ruta="data/raw/CDNOW_sample.txt"):
    """Lee CDNOW_sample.txt y une las compras del mismo día (regla 0.1)."""
    df = pd.read_csv(
        ruta, sep=r"\s+", header=None,
        names=["id_master", "id", "fecha", "cds", "monto"],
    )
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y%m%d")
    df = df.groupby(["id", "fecha"]).agg({"cds": "sum", "monto": "sum"}).reset_index()
    return df


def construir_resumen(df, fecha_fin_cal=FECHA_FIN_CAL):
    """Devuelve x, t_x, T (calibración) y x_star (holdout) por cliente."""
    cal = df[df["fecha"] <= fecha_fin_cal]
    hold = df[df["fecha"] > fecha_fin_cal]

    resumen = cal.groupby("id")["fecha"].agg(primera="min", ultima="max", n="count").reset_index()
    resumen["x"] = resumen["n"] - 1
    resumen["t_x"] = (resumen["ultima"] - resumen["primera"]).dt.days / 7
    resumen["T"] = (fecha_fin_cal - resumen["primera"]).dt.days / 7

    x_star = hold.groupby("id").size().rename("x_star")
    resumen = resumen.merge(x_star, on="id", how="left")
    resumen["x_star"] = resumen["x_star"].fillna(0).astype(int)
    return resumen